# Task 3.1 — YOLOv8 Dataset Preparation

In [ ]:
import random
from pathlib import Path
data_path = Path('../Data/archive/Train/')
classes = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
selected_images_path = []

for cls in classes:
    path = data_path/cls/'images'
    classe_images = list(path.iterdir())
    selected_classe_images = random.sample(classe_images, 2)
    selected_images_path.extend(selected_classe_images)

print(selected_images_path)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import random
from pathlib import Path

ARCHIVE = Path('../Data/archive')
CLASSES = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
COLORS  = [(255,0,0),(0,255,0),(0,0,255),(255,165,0)]

# Collect all (image, label) pairs from Train
pairs = []
for cls in CLASSES:
    for img_path in (ARCHIVE / 'Train' / cls / 'images').iterdir():
        lbl_path = ARCHIVE / 'Train' / cls / 'labels' / (img_path.stem + '.txt')
        if lbl_path.exists():
            pairs.append((img_path, lbl_path))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (img_path, lbl_path) in zip(axes.flat, random.sample(pairs, 8)):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for line in lbl_path.read_text().strip().splitlines():
        c, cx, cy, bw, bh = map(float, line.split())
        x1 = int((cx - bw/2) * w); y1 = int((cy - bh/2) * h)
        x2 = int((cx + bw/2) * w); y2 = int((cy + bh/2) * h)
        cv2.rectangle(img, (x1,y1), (x2,y2), COLORS[int(c)], 2)
        cv2.putText(img, CLASSES[int(c)], (x1,y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLORS[int(c)], 1)
    ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
import shutil
from pathlib import Path
import yaml

ARCHIVE = Path('../Data/archive')
OUT     = Path('../Data/yolo_dataset')

CLASSES = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
SPLITS  = ['Train', 'Val']

# Create flat output dirs
for split in SPLITS:
    for sub in ['images', 'labels']:
        (OUT / split.lower() / sub).mkdir(parents=True, exist_ok=True)

# Copy images and labels from per-class folders into flat dirs
for split in SPLITS:
    for cls in CLASSES:
        for sub in ['images', 'labels']:
            src = ARCHIVE / split / cls / sub
            dst = OUT / split.lower() / sub
            for f in src.iterdir():
                shutil.copy2(f, dst / f.name)

print('Done copying.')

In [ ]:
# Write dataset.yaml
cfg = {
    'path'  : str(OUT.resolve()),
    'train' : 'train/images',
    'val'   : 'val/images',
    'nc'    : 4,
    'names' : ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
}

yaml_path = OUT / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print(yaml_path.read_text())

In [ ]:
# Verify counts
for split in ['train', 'val']:
    imgs   = len(list((OUT / split / 'images').iterdir()))
    labels = len(list((OUT / split / 'labels').iterdir()))
    print(f'{split}: {imgs} images, {labels} labels')